# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- Use Pandas to load, inspect, and clean the dataset appropriately. 
- Transform relevant columns to create measures that address the problem at hand.
- **conduct EDA: visualization and statistical measures to understand the structure of the data**
- **recommend a set of manufacturers to consider as well as specific airplanes conforming to the client's request**
- **discuss the relationship between serious injuries/airplane damage incurred and at least *two* factors at play in the incident. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.**

In [1]:
# loading relevant packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Exploratory Data Analysis  
- Load in the cleaned data

In [2]:
# Load the cleaned dataset
df = pd.read_csv("data/AviationData_Cleaned.csv", low_memory=False)

# Restore the date column
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")

# Inspect the dataset
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nFirst five rows:")
display(df.head())

Dataset shape: (21447, 34)

Column names:
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date', 'Estimated.Occupants', 'Fatal.Serious.Injuries', 'Fatal.Serious.Injury.Fraction', 'Aircraft.Destroyed']

Data types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 21447 entries, 0 to 21446
Data columns (total 34 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Event.Id                       21447 non-null  str   

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Estimated.Occupants,Fatal.Serious.Injuries,Fatal.Serious.Injury.Fraction,Aircraft.Destroyed
0,20001214X42478,Incident,LAX83IA149B,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,NaN,588.0,VMC,Standing,Probable Cause,04-12-2014,NaN,NaN,NaN,0.0
1,20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,NaN,588.0,VMC,Taxi,Probable Cause,04-12-2014,NaN,NaN,NaN,0.0
2,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,...,NaN,NaN,IMC,Cruise,Probable Cause,02-05-2011,NaN,2.0,NaN,1.0
3,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,...,NaN,4.0,VMC,Standing,Probable Cause,17-10-2016,NaN,NaN,NaN,NaN
4,20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,NaN,NaN,NaN,NaN,...,NaN,289.0,VMC,Cruise,Probable Cause,01-02-2016,NaN,NaN,NaN,0.0


## Explore safety metrics across models/makes
- Remember that the client is interested in separate recommendations for smaller airplanes and larger airplanes. Choose a passenger threshold of 20 and separate the plane types. 

In [3]:
# Convert metrics to numeric in case the CSV reloaded them as strings
numeric_columns = [
    "Estimated.Occupants",
    "Fatal.Serious.Injury.Fraction",
    "Aircraft.Destroyed"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Keep records with a valid positive occupant estimate
safety_df = df.loc[
    df["Estimated.Occupants"].notna()
    & df["Estimated.Occupants"].gt(0)
].copy()

# Separate aircraft using the 20-occupant threshold
small_airplanes = safety_df.loc[
    safety_df["Estimated.Occupants"] <= 20
].copy()

large_airplanes = safety_df.loc[
    safety_df["Estimated.Occupants"] > 20
].copy()

print("All records:", len(df))
print("Records with valid occupant estimates:", len(safety_df))
print("Smaller-airplane records (20 or fewer):", len(small_airplanes))
print("Larger-airplane records (more than 20):", len(large_airplanes))

print("\nOccupant-count summary:")
display(
    safety_df.groupby(
        pd.cut(
            safety_df["Estimated.Occupants"],
            bins=[0, 20, float("inf")],
            labels=["Smaller (<=20)", "Larger (>20)"]
        ),
        observed=False
    )["Estimated.Occupants"].describe()
)

All records: 21447
Records with valid occupant estimates: 17562
Smaller-airplane records (20 or fewer): 16700
Larger-airplane records (more than 20): 862

Occupant-count summary:


,count,mean,std,min,25%,50%,75%,max
Estimated.Occupants,,,,,,,,
Smaller (<=20),16700.0,2.011557,1.632848,1.0,1.0,2.0,2.0,20.0
Larger (>20),862.0,143.080046,83.006272,21.0,80.0,138.5,181.0,576.0


#### Analyzing Makes

Explore the human injury risk profile for small and larger Makes:
- choose the 15 makes for each group possessing the lowest mean fatal/seriously injured fraction
- plot the mean fatal/seriously injured fraction for each of these subgroups side-by-side

In [ ]:
MIN_ACCIDENTS = 50

def get_top_15_makes(data):
    # Count all accident records for each make
    accident_counts = data.groupby("Make").size().rename("Accident.Count")

    # Calculate the mean injury fraction from records where it is available
    injury_means = (
        data.groupby("Make")["Fatal.Serious.Injury.Fraction"]
        .mean()
        .rename("Mean.Injury.Fraction")
    )

    # Combine the statistics
    make_summary = pd.concat(
        [accident_counts, injury_means],
        axis=1
    )

    # Retain makes with sufficient accident records and valid means
    make_summary = make_summary.loc[
        (make_summary["Accident.Count"] >= MIN_ACCIDENTS)
        & make_summary["Mean.Injury.Fraction"].notna()
    ]

    # Select the 15 makes with the lowest mean injury fraction
    return make_summary.sort_values(
        "Mean.Injury.Fraction",
        ascending=True
    ).head(15)


small_make_top15 = get_top_15_makes(small_airplanes)
large_make_top15 = get_top_15_makes(large_airplanes)

print("Top 15 makes — smaller airplanes:")
display(small_make_top15)

print("\nTop 15 makes — larger airplanes:")
display(large_make_top15)

Top 15 makes — smaller airplanes:


,Accident.Count,Mean.Injury.Fraction
Make,,
AVIAT AIRCRAFT INC,72,0.145833
BOEING,217,0.160374
GRUMMAN ACFT ENG COR-SCHWEIZER,58,0.206897
MAULE,144,0.209606
ROCKWELL INTERNATIONAL,51,0.217320
DIAMOND AIRCRAFT IND INC,74,0.222973
DEHAVILLAND,91,0.225754
STINSON,91,0.243590
AERONCA,149,0.252796



Top 15 makes — larger airplanes:


,Accident.Count,Mean.Injury.Fraction
Make,,
Boeing,61,0.049232
BOEING,425,0.051644
EMBRAER,64,0.062118
AIRBUS,99,0.091539


: 

**Distribution of injury rates: small makes**

Use a violinplot to look at the distribution of the fraction of passengers serious/fatally injured for small airplane makes. Just display makes with the ten lowest mean serious/fatal injury rates.

In [ ]:
# Minimum number of accident records per make
MIN_ACCIDENTS = 50

# Calculate accident counts and mean injury fractions
small_make_stats = (
    small_airplanes.groupby("Make")
    .agg({
        "Fatal.Serious.Injury.Fraction": ["count", "mean"]
    })
)

# Flatten the resulting column names
small_make_stats.columns = ["Valid.Injury.Count", "Mean.Injury.Fraction"]

# Add the total number of accident records for each make
small_make_stats["Accident.Count"] = (
    small_airplanes.groupby("Make").size()
)

# Keep makes with at least 50 accidents and a valid mean
small_make_stats = small_make_stats.loc[
    (small_make_stats["Accident.Count"] >= MIN_ACCIDENTS)
    & small_make_stats["Mean.Injury.Fraction"].notna()
]

# Select the 10 makes with the lowest mean injury fractions
small_make_top10 = small_make_stats.sort_values(
    "Mean.Injury.Fraction",
    ascending=True
).head(10)

# Select valid accident-level injury fractions for those makes
small_violin_df = small_airplanes.loc[
    small_airplanes["Make"].isin(small_make_top10.index)
    & small_airplanes["Fatal.Serious.Injury.Fraction"].notna(),
    ["Make", "Fatal.Serious.Injury.Fraction"]
].copy()

make_order = small_make_top10.index.tolist()

print("Selected smaller-airplane makes:")
display(small_make_top10)

# Plot the distributions
if small_violin_df.empty:
    print("No data available for the violin plot.")
else:
    plt.figure(figsize=(14, 7))

    sns.violinplot(
        data=small_violin_df,
        x="Make",
        y="Fatal.Serious.Injury.Fraction",
        order=make_order,
        inner="quartile",
        cut=0
    )

    plt.title(
        "Distribution of Fatal/Serious Injury Fractions: "
        "10 Smaller-Airplane Makes"
    )
    plt.xlabel("Aircraft Make")
    plt.ylabel("Fatal/Serious Injury Fraction")
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

Selected smaller-airplane makes:


,Valid.Injury.Count,Mean.Injury.Fraction,Accident.Count
Make,,,
AVIAT AIRCRAFT INC,72,0.145833,72
BOEING,217,0.160374,217
GRUMMAN ACFT ENG COR-SCHWEIZER,58,0.206897,58
MAULE,144,0.209606,144
ROCKWELL INTERNATIONAL,51,0.217320,51
DIAMOND AIRCRAFT IND INC,74,0.222973,74
DEHAVILLAND,91,0.225754,91
STINSON,91,0.243590,91
AERONCA,149,0.252796,149


**Distribution of injury rates: large makes**

Use a stripplot to look at the distribution of the fraction of passengers serious/fatally injured for large airplane makes. Just display makes with the ten lowest mean serious/fatal injury rates.

**Evaluate the rate of aircraft destruction for both small and large aircraft by Make.** 

Sort your results and keep the lowest 15.

#### Provide a short discussion on your findings for your summary statistics and plots:
- Make any recommendations for Makes here based off of the destroyed fraction and fraction fatally/seriously injured
- Comment on the calculated statistics and any corresponding distributions you have visualized.

### Analyze plane types
- plot the mean fatal/seriously injured fraction for both small and larger planes 
- also provide a distributional plot of your choice for the fatal/seriously injured fraction by airplane type (stripplot, violin, etc)  
- filter ensuring that you have at least ten individual examples in each model/make to average over

**Larger planes**

**Smaller planes**
- for smaller planes, limit your plotted results to the makes with the 10 lowest mean serious/fatal injury fractions

### Discussion of Specific Airplane Types
- Discuss what you have found above regarding passenger fraction seriously/ both small and large airplane models.

### Exploring Other Variables
- Investigate how other variables effect aircraft damage and injury. You must choose **two** factors out of the following but are free to analyze more:

- Weather Condition
- Engine Type
- Number of Engines
- Phase of Flight
- Purpose of Flight

For each factor provide a discussion explaining your analysis with appropriate visualization / data summaries and interpreting your findings.